# Milestone 4: Baselines, Feature Ablations, and Model Evaluation

This notebook evaluates pre-release features for forecasting Asian drama success (`target_pct` percentile within country-year cohort, and 3-class `target_tier`: Low < 25%, Medium 25-75%, High >= 75%).

### Evaluation Protocol & Methodological Guardrails
1. **Strict Temporal Split:** Train = dramas released $\le 2021$ ($N = 5,010$); Test = dramas released $\ge 2022$ ($N = 2,525$).
2. **Internal Validation Inside Train:** Model selection, hyperparameter tuning, and permutation importance are conducted strictly inside train ($\le 2019$ train with $N=3,520$ vs $2020–2021$ val with $N=1,490$). The test set is never touched during tuning.
3. **Primary Matrix:** Variant 2 (V2, static train-only priors) was declared as the headline benchmark before test evaluation and is kept as headline even though "all groups" scores slightly higher.
4. **Controlled Subsets Evaluated:**
   - **Full Test** ($N = 2,525$, High base rate: 25.3%)
   - **complete_info** ($N = 1,022$, High base rate: 38.9%): No missing core attributes.
   - **complete_info_strict** ($N = 635$, High base rate: 44.1%): `complete_info` AND content rating known AND aired_on known AND genres known.
   - **Per-Country Slices:** South Korea ($N=751$), Thailand ($N=790$), China ($N=532$), Japan ($N=452$).
5. **Statistical Rigor & Limitations:**
   - 500 bootstrap resamples on test metrics; 500 paired bootstrap resamples for model-vs-model comparisons.
   - *Note on Bootstrap Independence:* Bootstrap resamples treat dramas as independent observations although dramas share directors, cast, and sequels. Consequently, confidence intervals are likely somewhat narrow / optimistic.
   - *Multiple Testing Note:* Many comparisons were run, so results with an interval that barely excludes zero should be treated as suggestive rather than conclusive.
   - *Metric Reliability Note:* On `complete_info` and strict, macro-F1 is unstable because the Low class represents only ~7.5% to 9.5% of the data; ROC-AUC, PR-AUC, and Spearman correlation are the primary metrics there.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import pandas as pd
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

from src.modeling import (
    load_modeling_data,
    get_temporal_validation_split,
    get_expanding_window_folds,
    compute_metrics,
    bootstrap_evaluation,
    paired_bootstrap_comparison,
    MajorityClassBaseline,
    StratifiedRandomBaseline,
    CountryOnlyBaseline,
    PageCompletenessBaseline,
    HGB_TUNED_PARAMS,
    train_and_predict_hgb,
    train_and_predict_linear,
)
from src.features import get_feature_columns, FEATURE_GROUPS

print("Python executable:", sys.executable)
print(f"Scikit-learn version: {sklearn.__version__}")

# Load V2 Headline and V1 Deployment benchmarks
train_df, test_df = load_modeling_data("v2")
train_v1, test_v1 = load_modeling_data("v1")

for df in [train_df, test_df, train_v1, test_v1]:
    df["complete_info_strict"] = (
        (df["complete_info"] == 1) &
        (df["content_rating_is_missing"] == 0) &
        (df["aired_on_is_missing"] == 0) &
        (df["genre_is_missing"] == 0)
    ).astype(int)

print(f"Train dramas (<=2021): {len(train_df):,} | Test dramas (>=2022): {len(test_df):,}")
print(f"Test complete_info:        {int(test_df['complete_info'].sum()):,} ({test_df['complete_info'].mean():.1%})")
print(f"Test complete_info_strict: {int(test_df['complete_info_strict'].sum()):,} ({test_df['complete_info_strict'].mean():.1%})")

## Validation Protocol: Time-Ordered Folds Within Train

In [ ]:
tr_sub, val_sub = get_temporal_validation_split(train_df, val_cutoff_year=2019)
print(f"Internal Train (<=2019): {len(tr_sub):,} dramas")
print(f"Internal Val (2020-2021): {len(val_sub):,} dramas")

folds = get_expanding_window_folds(train_df)
print("\nExpanding-Window Folds inside Train:")
for f in folds:
    print(f"  Fold {f['fold']}: Train {f['train_years']} (N={f['train_size']:,}) -> Val {f['val_years']} (N={f['val_size']:,})")

## Step 1: Foundational Baseline Models

In [ ]:
y_train = train_df["target_tier"]

b_maj = MajorityClassBaseline().fit(train_df, y_train)
b_strat = StratifiedRandomBaseline(random_state=42).fit(train_df, y_train)
b_country = CountryOnlyBaseline(random_state=42).fit(train_df, y_train)
b_page = PageCompletenessBaseline(random_state=42).fit(train_df, y_train)

baselines = {
    "Majority Class (Medium)": b_maj,
    "Stratified Random": b_strat,
    "Country-Only": b_country,
    "Page Completeness Only": b_page,
}

def evaluate_baseline_dict(df_slice: pd.DataFrame) -> pd.DataFrame:
    y_true = df_slice["target_tier"].values
    t_pct = df_slice["target_pct"].values
    base_rate = (y_true == "High").mean()
    rows = []
    for name, model in baselines.items():
        preds = model.predict(df_slice)
        probs = model.predict_proba_high(df_slice)
        summary = bootstrap_evaluation(y_true, preds, probs, t_pct, n_boot=500, random_state=42)
        rows.append({
            "Baseline Model": name,
            "Accuracy": summary["Accuracy"]["str"],
            "Macro-F1": summary["Macro-F1"]["str"],
            "Balanced Acc": summary["Balanced Acc"]["str"],
            "ROC-AUC (High)": summary["ROC-AUC (High)"]["str"],
            f"PR-AUC [base:{base_rate:.3f}]": summary["PR-AUC (High)"]["str"],
            "Spearman": summary["Spearman"]["str"],
        })
    return pd.DataFrame(rows)

print("=== Step 1 Baselines: Full Test Set (N=2,525) ===")
display(evaluate_baseline_dict(test_df))

print("\n=== Step 1 Baselines: complete_info Subset (N=1,022) ===")
display(evaluate_baseline_dict(test_df[test_df["complete_info"] == 1].reset_index(drop=True)))

## Step 2: Feature Group Ablations & Predictive Modeling

In [ ]:
feature_sets = {
    "1. core": get_feature_columns(["core"]),
    "2. core + talent_reach (Headline)": get_feature_columns(["core", "talent_reach"]),
    "3. core + talent_reach + missingness": get_feature_columns(["core", "talent_reach", "missingness"]) + ["n_missing_credits"],
    "4. core + talent_reach + talent_exp": get_feature_columns(["core", "talent_reach", "talent_experience"]),
    "5. core + talent_reach + text": get_feature_columns(["core", "talent_reach", "text"]),
    "6. talent_reach only": get_feature_columns(["talent_reach"]),
    "7. all groups": get_feature_columns(["core", "talent_reach", "missingness", "talent_experience", "text"]) + ["n_missing_credits"],
}

predictions = {}

# Page completeness baseline
pc_clf = PageCompletenessBaseline(random_state=42).fit(train_df, train_df["target_tier"])
predictions["Page Completeness Baseline"] = (pc_clf.predict(test_df), pc_clf.predict_proba_high(test_df))

# 1-7. GBDT feature group ablations
for name, cols in feature_sets.items():
    predictions[name] = train_and_predict_hgb(train_df, test_df, cols, params=HGB_TUNED_PARAMS)

# 8. Drop 2015 burn-in
tr_no2015 = train_df[train_df["release_year"] >= 2016].reset_index(drop=True)
predictions["8. core + talent_reach (Drop 2015)"] = train_and_predict_hgb(
    tr_no2015, test_df, feature_sets["2. core + talent_reach (Headline)"], params=HGB_TUNED_PARAMS
)

# 9. V1 vs V2 priors
predictions["9. core + talent_reach (V1 rolling)"] = train_and_predict_hgb(
    train_v1, test_v1, feature_sets["2. core + talent_reach (Headline)"], params=HGB_TUNED_PARAMS
)

# 10. Linear Reference (Logistic Regression on Headline)
predictions["10. Linear Reference (core + reach)"] = train_and_predict_linear(
    train_df, test_df, feature_sets["2. core + talent_reach (Headline)"]
)

print(f"Trained all {len(predictions)} model variants successfully!")

In [ ]:
def evaluate_models_slice(mask: np.ndarray, slice_title: str) -> pd.DataFrame:
    y_true = test_df.loc[mask, "target_tier"].values
    t_pct = test_df.loc[mask, "target_pct"].values
    base_rate = float((y_true == "High").mean())
    n = len(y_true)
    
    rng = np.random.RandomState(42)
    boot_indices = [rng.choice(n, size=n, replace=True) for _ in range(500)]
    
    rows = []
    for model_name, (pred, prob) in predictions.items():
        p_sub = pred[mask]
        pr_sub = prob[mask]
        pt = compute_metrics(y_true, p_sub, pr_sub, t_pct)
        
        boot_list = {k: [] for k in pt.keys()}
        for b_idx in boot_indices:
            b_m = compute_metrics(y_true[b_idx], p_sub[b_idx], pr_sub[b_idx], t_pct[b_idx])
            for k, v in b_m.items():
                boot_list[k].append(v)
                
        def get_ci_str(metric_name: str, fmt: str = ".3f") -> str:
            val = pt[metric_name]
            arr = np.array(boot_list[metric_name])
            arr_valid = arr[~np.isnan(arr)]
            if len(arr_valid) == 0 or np.isnan(val):
                return "N/A"
            low = np.percentile(arr_valid, 2.5)
            high = np.percentile(arr_valid, 97.5)
            return f"{val:{fmt}} [{low:{fmt}}, {high:{fmt}}]"
            
        rows.append({
            "Model": model_name,
            "Accuracy": get_ci_str("Accuracy"),
            "Macro-F1": get_ci_str("Macro-F1"),
            "Balanced Acc": get_ci_str("Balanced Acc"),
            "ROC-AUC (High)": get_ci_str("ROC-AUC (High)"),
            f"PR-AUC [base:{base_rate:.3f}]": get_ci_str("PR-AUC (High)"),
            "Spearman": get_ci_str("Spearman"),
        })
    return pd.DataFrame(rows)

print("=== Full Test Set (N=2,525) ===")
display(evaluate_models_slice(np.ones(len(test_df), dtype=bool), "Full Test Set"))

print("\n=== complete_info Subset (N=1,022) ===")
display(evaluate_models_slice(test_df["complete_info"].values == 1, "complete_info Subset"))

print("\n=== complete_info_strict Subset (N=635) ===")
display(evaluate_models_slice(test_df["complete_info_strict"].values == 1, "complete_info_strict Subset"))

## Step 3: Probability Calibration (Fitted on Internal Validation Only)

In [ ]:
# Calibration: Fit on internal validation (<=2019 train vs 2020-2021 val)
headline_cols = feature_sets["2. core + talent_reach (Headline)"]
y_val_bin = (val_sub["target_tier"].values == "High").astype(int)

# Predict on val
_, val_probs = train_and_predict_hgb(tr_sub, val_sub, headline_cols, params=HGB_TUNED_PARAMS)

# Fit Platt and Isotonic calibrators
platt = LogisticRegression(random_state=42).fit(val_probs.reshape(-1, 1), y_val_bin)
iso = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip").fit(val_probs, y_val_bin)

# Apply to test
_, test_probs_raw = predictions["2. core + talent_reach (Headline)"]
test_probs_platt = platt.predict_proba(test_probs_raw.reshape(-1, 1))[:, 1]
test_probs_iso = iso.predict(test_probs_raw)

y_test_bin = (test_df["target_tier"].values == "High").astype(int)
base_rate = float(y_test_bin.mean())
brier_baseline = base_rate * (1 - base_rate)

def eval_cal(probs, name):
    brier = float(np.mean((probs - y_test_bin)**2))
    bins = np.linspace(0.0, 1.0, 11)
    bin_ids = np.digitize(probs, bins) - 1
    ece = 0.0
    rows = []
    for b in range(10):
        mask = bin_ids == b
        nb = int(mask.sum())
        if nb == 0:
            continue
        mp = float(probs[mask].mean())
        mo = float(y_test_bin[mask].mean())
        ece += (nb / len(probs)) * abs(mp - mo)
        rows.append({"Bin": f"[{bins[b]:.1f}, {bins[b+1]:.1f})", "N": nb, "Pred": f"{mp:.3f}", "Obs": f"{mo:.3f}", "Diff": f"{mp-mo:+.3f}"})
    return brier, ece, pd.DataFrame(rows)

brier_raw, ece_raw, t_raw = eval_cal(test_probs_raw, "Raw")
brier_platt, ece_platt, t_platt = eval_cal(test_probs_platt, "Platt")
brier_iso, ece_iso, t_iso = eval_cal(test_probs_iso, "Isotonic")

print(f"Brier baseline (predicting base rate {base_rate:.3f}): {brier_baseline:.4f}")
print(f"Raw Model:   Brier = {brier_raw:.4f}, ECE = {ece_raw:.4f}")
print(f"Platt Scaled: Brier = {brier_platt:.4f}, ECE = {ece_platt:.4f}")
print(f"Isotonic:     Brier = {brier_iso:.4f}, ECE = {ece_iso:.4f}")
display(t_iso)